# 실제 녹음 학습 확인 실험 — GPU 서버 실행 (S15P31A307-164, 설계 CHART-003-real-drum-data-check)

학습 데이터에 실제 녹음 드럼이 없어서 ENST 탐이 떨어졌는지 확인한다. 학습 4개를 GPU 6번에서 **동시에** 돌린다.

- `tom_6cls_real_no_d1`·`_d2`·`_d3`: `tom_6cls` 데이터에 ENST 드러머 둘의 전체 연주와 MDB 두 묶음을 더함. 뺀 드러머·묶음으로 평가
- `tom_6cls_plain`: `tom_6cls`와 데이터는 같고 탐 가중치 2배·탐 중심 윈도만 끔

**준비물.** 로컬 `AI/`에서 `uv run python scripts/pack_server_upload.py --code-only`로 만든 `data/upload/server/S15P31A307_code.zip`을 JupyterHub `ssaplash/upload/`에, 이 파일을 `ssaplash/notebooks/`에 올린다. 서버에는 `.venv`·데이터 링크(`data/e-gmd`, `data/star_drums/full`, `data/enst`, `data/gmd`, `data/mdb_drums`)·`data/rule_features/`(학습 전 가중치 특징)·`data/models/adtof_ft/tom_6cls`·`data/results/rule-eval-base-pedal`·`data/results/rule-eval-ft-tom_6cls`가 이미 있다.

**순서.** 1 설정 → 1-1 지금 상태(여러 번) → 2 코드 덮어 풀기 → 3 데이터 점검 → 4 짧은 시험 → 5 학습 4개 동시 시작(백그라운드, 한 번만) → 6 진행 확인(여러 번) → 7 규칙 포함 특징 저장(학습이 끝난 것부터, 여러 번 쳐도 됨) → 8 진행 확인 → 9 rule-eval → 9-1 확인 → 10 모델 출력 점수 → 11 drummer_1 탐 오검출 자리 → 12 채점표 → 13 결과 묶기.

**끊기면.** 5·7·9·10번은 백그라운드라 브라우저를 닫아도 끝까지 돈다. 커널이 다시 시작됐으면 1번 → 1-1번을 치고 이어 간다. 학습이 '멈춤'이면 5번을 다시 치면 `last.pth`에서 이어 간다. 학습은 `~/ssaplash/runs/164-real-check`의 코드 사본으로 돌아서, 학습 중에 2번 셀로 코드를 다시 풀어도 도는 학습은 안 바뀐다.

**Jupyter가 버벅이면.** 6번 셀 맨 아래 주석을 풀어 학습을 끄고, 1번 셀 `PARALLEL`을 2로 바꿔 1번 → 5번을 다시 친다. 끈 학습은 이어 간다.

In [ ]:
# 1. 설정. 지정받은 GPU는 6번
import getpass
import os
import sys

GPU_ID = "6"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"   # nohup 로그가 바로바로 써지게
os.environ["MPLBACKEND"] = "Agg"
# 데이터 읽기 작업 하나가 코어 1개만 쓰게 함. 안 하면 numpy가 작업마다 서버 코어 256개를 다 쓰려 해서 서로 뺏음
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
PY = sys.executable
VENV_PY = f"{AI}/.venv/bin/python"
ME = getpass.getuser()
# 학습 실행 폴더. 코드 사본과 대기열 상태 파일(status/)이 여기 생김. 코드를 바꿔 새로 돌릴 때만 이름을 바꿈
RUN_DIR = f"{WORK}/runs/164-real-check"
QUEUE = f"{VENV_PY} {AI}/scripts/train_queue.py"
UPLOAD = f"{WORK}/upload"
RESULTS = f"{WORK}/results"
PARALLEL = 4          # 동시에 돌릴 학습 수. Jupyter가 버벅이면 2
CPUS = "0-95"         # 학습·데이터 읽기 작업을 묶을 코어. 나머지 코어는 Jupyter·다른 사용자 몫
NUM_WORKERS = 8       # 학습마다 데이터 읽기 작업 수. 넷이면 32개
RC = "configs/bench/transcribers/adtof_drumsep/real_check"
# 학습 이름 → (학습 설정, 규칙 포함 평가용 어댑터 설정)
RUNS = {
    "tom_6cls_real_no_d1": ("configs/train/real_check/tom_6cls_real_no_d1.yaml", f"{RC}/adtof_drumsep_tom_6cls_real_no_d1.yaml"),
    "tom_6cls_plain": ("configs/train/real_check/tom_6cls_plain.yaml", f"{RC}/adtof_drumsep_tom_6cls_plain.yaml"),
    "tom_6cls_real_no_d2": ("configs/train/real_check/tom_6cls_real_no_d2.yaml", f"{RC}/adtof_drumsep_tom_6cls_real_no_d2.yaml"),
    "tom_6cls_real_no_d3": ("configs/train/real_check/tom_6cls_real_no_d3.yaml", f"{RC}/adtof_drumsep_tom_6cls_real_no_d3.yaml"),
}
OUT = {r: f"data/models/adtof_ft/{r}" for r in RUNS}
FEAT = {r: f"data/rule_features_rc_{r}" for r in RUNS}
FEAT_LOG = {r: f"{AI}/rule_features_rc_{r}.log" for r in RUNS}
RES = "data/results/real_check"
DS = "configs/bench/datasets"
# 규칙 포함 채점 데이터. E-GMD test 1,170곡은 시간이 커서 모델 출력 점수(10번)에서만 봄
EVAL_DS = f"{DS}/enst_minus_one.yaml {DS}/mdb_drums.yaml {DS}/gmd_test.yaml {DS}/star_drums_test.yaml"
DS_NAMES = ("enst_minus_one", "mdb_drums", "gmd_test", "star_drums_test")
DS_SONGS = {"enst_minus_one": 64, "mdb_drums": 23, "gmd_test": 45, "star_drums_test": 31}
os.makedirs(RESULTS, exist_ok=True)
%cd $AI
os.makedirs(RES, exist_ok=True)
print("CPU 코어", os.cpu_count(), "· 동시 학습", PARALLEL, "· 코어", CPUS, "· 학습마다 데이터 읽기 작업", NUM_WORKERS)
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!df -h $HOME | tail -1
!ls -la $UPLOAD/S15P31A307_code.zip
# 확인: 'CPU 코어 256 · 동시 학습 4 · 코어 0-95 · 학습마다 데이터 읽기 작업 8', GPU 한 줄(H200).
#       memory.used가 100,000MiB를 넘으면 다른 사용자가 크게 쓰는 중이니 기다림. 코드 zip 날짜가 오늘 올린 것

In [ ]:
# 1-1. 지금 상태 (여러 번 쳐도 됨). 다른 셀을 치기 전에 먼저 침
!$QUEUE status --run-dir $RUN_DIR
!ps -u $ME -o pid,etime,pcpu,args | grep -E "[a]dtof-finetune|[t]rain_queue|[r]ule-features|[r]ule-eval|[a]dtof-eval" | cut -c1-150
!nvidia-smi -i $GPU_ID --query-gpu=memory.used,utilization.gpu --format=csv,noheader
# 확인: 대기열을 띄우기 전이면 '대기열을 시작한 적 없음: …/runs/164-real-check/status'.
#   띄운 뒤에는 '[대기열] 도는 중 pid …', '[서버] load … / 코어 256, 남은 메모리 …GB / …GB, 우리 데이터 읽기 작업 32개',
#   학습마다 '== tom_6cls_real_no_d1: 도는 중 pid …' / '끝남' / '실패 (종료 코드 …)' / '멈춤 (…)'.
#   ps 줄은 우리 계정에서 도는 학습·특징 저장·채점

In [ ]:
# 2. 코드 덮어 풀기. 기존 폴더를 지우지 않고 위에 덮어씀 (.venv·데이터 링크·특징 파일이 그대로 남음)
!$PY -m zipfile -e $UPLOAD/S15P31A307_code.zip $WORK
%cd $AI
!ls configs/train/real_check/ configs/bench/datasets/real_check/ | head -30
!ls scripts/score_heldout.py $RC
!grep -c "parallel" scripts/train_queue.py
!grep -c "ignore_classes" src/ssaplash_ai/train/adtof_data.py
!.venv/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
# 확인: configs/train/real_check/ 아래 folds.yaml과 tom_6cls_* 8개, datasets/real_check/ 아래 yaml 16개,
#       score_heldout.py와 어댑터 yaml 8개, 숫자 두 개는 0이 아님(새 코드가 풀림), 'torch 2.8.0+cu128 cuda True'.
#       숫자가 0이면 zip이 예전 것. 로컬에서 zip을 다시 만들어 올림. 'cuda False'면 1번 셀을 다시 침

In [ ]:
# 3. 데이터 점검 (여러 번 쳐도 됨, 1~2분). 새 yaml의 곡 수·누락과 ENST 드러머별 분량
!ls -la data/enst data/mdb_drums 2>&1 | head -6
!.venv/bin/ssaplash-bench check-data $DS/real_check/enst_minus_one_d1.yaml $DS/real_check/enst_minus_one_d2.yaml $DS/real_check/enst_minus_one_d3.yaml $DS/real_check/mdb_fold1.yaml $DS/real_check/mdb_fold2.yaml $DS/real_check/mdb_fold3.yaml $DS/real_check/mdb_train_no_f1.yaml $DS/real_check/mdb_train_no_f2.yaml $DS/real_check/mdb_train_no_f3.yaml $DS/real_check/enst_train_no_d1.yaml $DS/real_check/enst_train_no_d2.yaml $DS/real_check/enst_train_no_d3.yaml
import subprocess

ENST_COUNT = '''
import collections, soundfile as sf
from ssaplash_ai.bench.datasets import load_dataset
from ssaplash_ai.bench.datasets.enst import song_kind, parse_song_id
d = load_dataset('configs/bench/datasets/real_check/enst_train_no_d1.yaml'); d.config.extra['drummers'] = [1, 2, 3]
n, sec, tom = collections.Counter(), collections.Counter(), collections.Counter()
for s in d.song_ids():
    k = (parse_song_id(s)[0], song_kind(parse_song_id(s)[1]))
    n[k] += 1; sec[k] += sf.info(str(d._paths(s)['drums'])).duration
    h, _ = d.hits_of(s); tom[k] += int(sum(str(p).endswith('_TOM') for p in h['pad']))
for k in sorted(n): print(f'드러머 {k[0]} {k[1]:15s} {n[k]:4d}곡 {sec[k]/60:6.1f}분 탐 {tom[k]:5d}')
print('합계', sum(n.values()), '곡', round(sum(sec.values())/60, 1), '분, 탐', sum(tom.values()))
'''
out = subprocess.run([VENV_PY, "-c", ENST_COUNT], cwd=AI, capture_output=True, text=True, env=dict(os.environ))
print(out.stdout or out.stderr[-2000:])
# 확인: enst_minus_one_d1·d2·d3 21·22·21곡, mdb_fold1·2·3 8·7·8곡, mdb_train_no_f1·2·3 15·16·15곡, 전부 누락 0.
#       enst_train_no_d* 는 기대 곡 수 없이 곡 수만 나옴(드러머 둘의 전체).
#       아래 표는 드러머 1·2·3 × 종류(hits·phrase·solo·minus-one·MIDI-minus-one)별 곡 수·분·탐 타격 수와 합계.
#       이 표를 세션에 붙여 줌(설계 문서에 ENST 분량으로 적음). 합계는 주석 파일 318개 근처여야 함.
#       누락이 있거나 0곡이면 `ls data/enst/ENST-drums-public/drummer_1/annotation | head` 결과를 적어 두고 멈춤

In [ ]:
# 4. 짧은 시험 (학습마다 20스텝 × 1에폭, 하나에 2~3분). 데이터 읽기·손실 마스크·검증·문턱값 맞추기가 끝까지 도는지만 봄
for r, (cfg, _) in RUNS.items():
    if os.path.isfile(f"{OUT[r]}/last.pth"):
        print(r, "본 학습 가중치(last.pth)가 이미 있어 시험 건너뜀")
        continue
    smoke = f"data/models/adtof_ft/smoke_{r}"
    !rm -rf $smoke
    !.venv/bin/python -m ssaplash_ai.train.cli adtof-finetune --config $cfg --out $smoke --max-epochs 1 --steps-per-epoch 20 --num-workers 4 2>&1 | grep -E "\[train\]|Error|error"
    !cut -d, -f1-8 $smoke/log.csv
# 확인 (학습마다):
#   tom_6cls_plain: '[train] 학습 곡 27329개(해시 71c8c3a549f1), 검증 곡 40개, 클래스 five_pedal(6), 장치 cuda' (tom_6cls와 같은 해시)
#   tom_6cls_real_no_d*: 학습 곡 수가 27329 + ENST 드러머 둘 + MDB 15·16곡만큼 많고 해시가 다름
#   '[train] 기준값(기본 가중치 5클래스): kick 0.952, snare 0.812, tom 0.407, hihat 0.775, cymbal 0.690, 페달 0.000 (…)'
#   '[train] 에폭 -1 …', '[train] 에폭 0 …', '[train] 문턱값 맞춤 […]', '[train] 끝.'. log.csv는 세 줄
#   에폭 줄 끝의 '(기준값보다 0.02 넘게 떨어진 클래스 있음, 선택 안 함)'과 '경고: 기준값 조건을 넘는 에폭이 없어'는 시험에서는 정상
#   'ignore_classes' 오류나 'KeyError'가 나면 전문을 적어 두고 멈춤

In [ ]:
# 5. 학습 4개 동시 시작 (백그라운드, 한 번만. 다시 쳐도 대기열이 돌고 있으면 안 띄움)
#    RUN_DIR에 코드 사본을 풀어 그 사본으로 학습. 학습과 데이터 읽기 작업은 CPUS 코어에만 묶임. 끊긴 학습은 last.pth에서 이어 감
runs = " ".join(f"--run {r}={cfg}" for r, (cfg, _) in RUNS.items())
!$QUEUE start --run-dir $RUN_DIR --zip $UPLOAD/S15P31A307_code.zip --shared-ai $AI --python $VENV_PY --num-workers $NUM_WORKERS --parallel $PARALLEL --cpus $CPUS $runs
# 확인 (처음): '코드 사본 풂: /home/j-k15a307/ssaplash/runs/164-real-check/S15P31A307/AI (파일 6xx개)',
#              'data 링크: …/runs/164-real-check/S15P31A307/AI/data → /home/j-k15a307/ssaplash/S15P31A307/AI/data',
#              '대기열 시작 pid 숫자: tom_6cls_real_no_d1 → tom_6cls_plain → tom_6cls_real_no_d2 → tom_6cls_real_no_d3, 동시 4개, 데이터 읽기 작업 학습마다 8개, nice 19, 코어 0-95'
#   대기열이 이미 돌면 '대기열이 이미 돌고 있음 pid …'. 그대로 6번으로
#   '학습 설정을 못 찾음'이면 코드 zip이 예전 것. 2번 셀 확인
#   'taskset: command not found'가 6번 로그에 나오면 1번 셀 CPUS를 "" 로 바꾸고 6번 맨 아래 stop → 5번 다시

In [ ]:
# 6. 진행 확인 (여러 번 쳐도 됨). 학습 하나에 약 1~1.5시간
!$QUEUE status --run-dir $RUN_DIR
!nvidia-smi -i $GPU_ID --query-gpu=memory.used,utilization.gpu --format=csv,noheader
!taskset -cp $(pgrep -u $ME -f "[a]dtof-finetune" | head -1) 2>/dev/null || true
# 확인: '[대기열] 도는 중 pid …', '[서버] load …, 우리 데이터 읽기 작업 32개'.
#   학습마다 '[train] 에폭 N 손실 … 검증 f1 … 탐 … 페달 …' 줄과 'log.csv: N,…' 줄이 늘어남. 에폭 하나에 4~8분.
#   10/30 tom_6cls 비교값: 에폭 7에서 검증 탐 0.7대, 에폭 11에서 끝남
#   맨 아래 줄은 "pid …'s current affinity list: 0-95" 여야 코어 묶기가 된 것
#   끝난 학습은 '끝남'. 그 학습부터 7번 셀을 쳐도 됨
#   '실패 (종료 코드 …)'면 RUN_DIR/status/<이름>.log 끝을 적어 둠. 다른 학습은 계속 돎
#   '멈춤'은 끝 표시 없이 꺼진 것. 5번 셀을 다시 치면 그 자리에서 이어 감
#   Jupyter가 버벅이면 아래 줄 주석을 풀어 끄고(이어 학습 가능), 1번 셀 PARALLEL을 2로 바꿔 1번 → 5번을 다시 침
# !$QUEUE stop --run-dir $RUN_DIR

In [ ]:
# 7. 규칙 포함 점수용 특징 저장 (백그라운드, nice 19). 대기열에서 끝남 표시가 난 학습만 시작, 이미 시작한 것은 건너뜀. 여러 번 쳐도 됨
#    분리 트랙은 data/rule_features(학습 전 가중치 특징)에서 복사하고 채보 모델만 새 가중치로 돌림
import subprocess

for r, (_, model_yaml) in RUNS.items():
    if os.path.exists(FEAT_LOG[r]):
        print(r, "이미 시작했음, 건너뜀")
        continue
    if not (os.path.isfile(f"{RUN_DIR}/status/{r}.DONE") and os.path.isfile(f"{OUT[r]}/best.pth")):
        print(r, "학습이 아직 안 끝남, 건너뜀 (6번 셀로 확인 뒤 다시 침)")
        continue
    cmd = ["nice", "-n", "19", ".venv/bin/ssaplash-bench", "rule-features", "--model", model_yaml,
           "--out", FEAT[r], "--stems-from", "data/rule_features", "--dataset", *EVAL_DS.split()]
    proc = subprocess.Popen(["nohup", *cmd], cwd=AI, stdout=open(FEAT_LOG[r], "w"), stderr=subprocess.STDOUT,
                            env=dict(os.environ), start_new_session=True)
    print("시작.", r, "pid", proc.pid, "로그", FEAT_LOG[r])
# 확인: 끝난 학습마다 '시작. tom_6cls_… pid …'. 안 끝난 것은 '… 건너뜀'이라 나중에 다시 침

In [ ]:
# 8. 특징 저장 진행 확인 (여러 번). 곡 수가 64 / 23 / 45 / 31이 되면 끝
import glob

for r in RUNS:
    print("=====", r)
    if not os.path.exists(FEAT_LOG[r]):
        print("아직 시작 안 함 (학습이 끝나면 7번 셀)")
        continue
    print(" ".join(f"{d} {len(glob.glob(f'{FEAT[r]}/{d}/**/*.npz', recursive=True))}" for d in DS_NAMES))
    !tail -1 {FEAT_LOG[r]}
    !grep -c "분리 트랙 재사용" {FEAT_LOG[r]} || true
    !grep -c '실패 [1-9]' {FEAT_LOG[r]} || true
# 확인: 'enst_minus_one 64 mdb_drums 23 gmd_test 45 star_drums_test 31'이 되면 끝. 학습 하나에 10~30분.
#   '분리 트랙 재사용' 수가 저장한 곡 수와 같아야 함(0이면 분리를 새로 돌리는 중이라 몇 시간 걸림. 멈추고 적어 둠). 실패 줄 수는 0

In [ ]:
# 9. rule-eval (CPU, 백그라운드, nice 19). 특징 저장이 끝난 학습만 띄움. 여러 번 쳐도 됨
#    작업 하나가 네 데이터의 특징을 메모리에 올려서, 남은 메모리의 70% 안에서만 띄우고 나머지는 9-1번이 나중에 띄움
import glob
import subprocess

CAND = "docs/chart-rules/tom-newkits-2026-10-29/candidates.yaml"
RULE_LOG = {r: f"{AI}/rule_eval_rc_{r}.log" for r in RUNS}


def pid_alive(pid):
    try:
        with open(f"/proc/{pid}/stat") as f:
            return f.read().rsplit(")", 1)[1].split()[0] != "Z"
    except (OSError, IndexError):
        return False


def rule_state(r):
    if not os.path.exists(RULE_LOG[r]):
        return "대기"
    text = open(RULE_LOG[r], encoding="utf-8", errors="replace").read()
    if "[rule-eval] 결과 →" in text:
        return "끝남"
    try:
        pid = int(open(RULE_LOG[r] + ".pid").read())
    except (OSError, ValueError):
        return "실패"
    return "도는 중" if pid_alive(pid) else "실패"


def feats_done(r):
    return all(len(glob.glob(f"{FEAT[r]}/{d}/**/*.npz", recursive=True)) >= n for d, n in DS_SONGS.items())


def mem_avail_gb():
    for line in open("/proc/meminfo"):
        if line.startswith("MemAvailable"):
            return int(line.split()[1]) / 1024**2
    return 0.0


def job_mem_gb(feats):
    # 스템이 파일에는 float16, 메모리에는 float32라 파일 크기의 약 2.2배
    return sum(os.path.getsize(p) for d in DS_NAMES for p in glob.glob(f"{feats}/{d}/**/*.npz", recursive=True)) * 2.2 / 1e9


def launch_rule_jobs():
    avail = mem_avail_gb() * 0.7
    for r in RUNS:
        if rule_state(r) != "대기":
            continue
        if not feats_done(r):
            print(f"{r}: 특징 저장이 아직 안 끝남 (8번 셀)")
            continue
        need = job_mem_gb(FEAT[r])
        if need > avail:
            print(f"{r}: 메모리가 모자라 대기 (필요 약 {need:.0f}GB, 쓸 수 있는 것 {avail:.0f}GB)")
            continue
        cmd = ["nice", "-n", "19", ".venv/bin/ssaplash-bench", "rule-eval", "--features", FEAT[r],
               "--dataset", *EVAL_DS.split(), "--candidates", CAND, "--out", f"{RES}/rule-eval-{r}",
               "--two-tom-upper", "MID_TOM", "--mlflow-uri", "none"]
        proc = subprocess.Popen(["nohup", *cmd], cwd=AI, stdout=open(RULE_LOG[r], "w"), stderr=subprocess.STDOUT,
                                env=dict(os.environ), start_new_session=True)
        with open(RULE_LOG[r] + ".pid", "w") as f:
            f.write(str(proc.pid))
        avail -= need
        print(f"시작. {r} pid {proc.pid} (메모리 약 {need:.0f}GB)")


print(f"[서버] 남은 메모리 {mem_avail_gb():.0f}GB")
launch_rule_jobs()
# 확인: 특징 저장이 끝난 학습마다 '시작. tom_6cls_… pid … (메모리 약 NGB)'. 나머지는 '특징 저장이 아직 안 끝남' 또는 '메모리가 모자라 대기'.
#   다시 쳐도 이미 돌거나 끝난 작업은 안 띄움. 실패한 작업을 다시 돌리려면 그 로그(AI/rule_eval_rc_<이름>.log)를 지우고 침

In [ ]:
# 9-1. rule-eval 진행 확인 (여러 번). 대기한 작업도 여기서 띄움
for r in RUNS:
    st = rule_state(r)
    text = open(RULE_LOG[r], encoding="utf-8", errors="replace").read() if os.path.exists(RULE_LOG[r]) else ""
    print(f"{r:22s} {st:5s} 특징 읽은 데이터 {text.count('특징 읽음')}/{len(DS_NAMES)}, 채점 {text.count('f1Tom=')}/{5 * len(DS_NAMES)}")
    if st == "실패":
        print("    로그 끝:", *text.strip().splitlines()[-3:], sep="\n    ")
launch_rule_jobs()
for r in RUNS:
    if rule_state(r) == "끝남":
        print("==", r)
        !grep pk_gap30 {RES}/rule-eval-{r}/candidates.csv | cut -d, -f1,2,11,12,19,23
# 확인: 학습마다 한 줄. '끝남'이면 아래에 pk_gap30 네 줄(후보, 데이터셋, f1Pedal, precPedal, f1Tom, tomFalseKept). 작업 하나에 20~60분.
#   '실패'면 로그 끝 3줄을 붙여 줌. 4개가 다 '끝남'이면 12번(채점표)

In [ ]:
# 10. 모델 출력만 본 점수 (GPU, 백그라운드, 한 번만, 30~60분). 4개 학습이 다 끝난 뒤 침
#     학습마다 안 본 ENST 드러머·MDB 묶음을 그 학습·tom_6cls·기본 가중치로, 합성 데이터 셋은 네 학습 모두로
import subprocess

assert all(os.path.isfile(f"{RUN_DIR}/status/{r}.DONE") for r in RUNS), "아직 안 끝난 학습이 있음 (6번 셀)"
EVAL_LOG = f"{AI}/adtof_eval_rc.log"
assert not os.path.exists(EVAL_LOG), "이미 시작했음. 10-1번 셀로 확인"
T6 = "data/models/adtof_ft/tom_6cls/best.pth"
cli = ".venv/bin/python -m ssaplash_ai.train.cli adtof-eval"
steps = []
for k in (1, 2, 3):
    r = f"tom_6cls_real_no_d{k}"
    steps.append(f"{cli} --checkpoint {OUT[r]}/best.pth {T6} --dataset {DS}/real_check/enst_minus_one_d{k}.yaml {DS}/real_check/mdb_fold{k}.yaml --out {RES}/adtof-eval-heldout-d{k}")
held = " ".join(f"{DS}/real_check/enst_minus_one_d{k}.yaml {DS}/real_check/mdb_fold{k}.yaml" for k in (1, 2, 3))
steps.append(f"{cli} --dataset {held} --out {RES}/adtof-eval-heldout-base")
steps.append(f"{cli} --checkpoint {OUT['tom_6cls_plain']}/best.pth --dataset {held} --out {RES}/adtof-eval-heldout-plain")
cks = " ".join(f"{OUT[r]}/best.pth" for r in RUNS)
steps.append(f"{cli} --checkpoint {cks} --dataset {DS}/gmd_test.yaml {DS}/star_drums_test.yaml {DS}/egmd_test_acoustic.yaml --out {RES}/adtof-eval-synth")
proc = subprocess.Popen(["nohup", "nice", "-n", "19", "bash", "-c", " && ".join(steps)], cwd=AI,
                        stdout=open(EVAL_LOG, "w"), stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
print("시작. pid", proc.pid, "로그", EVAL_LOG)
# 확인: '시작. pid 숫자'. 다시 치면 assert에 걸림(정상)

In [ ]:
# 10-1. 모델 출력 점수 진행 확인 (여러 번)
!grep -c "\[adtof-eval\].*f1" $AI/adtof_eval_rc.log
!tail -2 $AI/adtof_eval_rc.log
!ls $RES | grep adtof-eval
# 확인: 숫자는 끝나면 36(안 본 곡 2체크포인트×2데이터×3 = 12, 기본 가중치 6, plain 6, 합성 4×3 = 12). 숫자가 36이 아니어도 마지막 줄에 'summary.md'가 있고
#       ls에 adtof-eval-heldout-d1·d2·d3·-base·-plain·adtof-eval-synth 6개가 있으면 끝). 'Traceback'이 있으면 끝 20줄을 적어 둠

In [ ]:
# 11. ENST drummer_1 정답에 없는 탐 자리 (CPU, nice 19, 5~15분). tom_6cls_real_no_d1·tom_6cls_plain의 특징 저장이 끝난 뒤
#     정답 탐이 없는데 남은 탐 예측마다 ±50ms 안 ENST 원래 주석(bd 킥·sd 스네어·chh 하이햇 등)을 셈
feats = " ".join(["--features base=data/rule_features", "--features tom_6cls=data/rule_features_ft_tom_6cls",
                  f"--features tom_6cls_plain={FEAT['tom_6cls_plain']}", f"--features tom_6cls_real_no_d1={FEAT['tom_6cls_real_no_d1']}"])
!nice -n 19 .venv/bin/python scripts/check_false_toms.py $feats --dataset $DS/enst_minus_one.yaml --song-prefix drummer_1 --candidates docs/chart-rules/tom-newkits-2026-10-29/candidates.yaml --candidate pk_gap30 --two-tom-upper MID_TOM --out $RES/false-toms-enst-drummer_1.md
# 확인: 곡마다 '[false-toms] drummer_1/…: base 숫자, tom_6cls 숫자, tom_6cls_plain 숫자, tom_6cls_real_no_d1 숫자' 21줄, 끝에 '결과 →' 한 줄.
#   tom_6cls 숫자를 다 더하면 3,268(10/31과 같음). 'No such file'이면 data/rule_features_ft_tom_6cls가 지워진 것이니 적어 둠

In [ ]:
# 12. 채점표 (CPU, 몇 초). 학습마다 안 본 곡만 모아 기본 가중치·tom_6cls·tom_6cls_plain과 같은 곡으로 비교하고 판정 줄을 붙임
runs_arg = " ".join(f"--run {r}={RES}/rule-eval-{r}" for r in RUNS if r != "tom_6cls_plain")
cmp_arg = (f"--compare base=data/results/rule-eval-base-pedal --compare tom_6cls=data/results/rule-eval-ft-tom_6cls "
           f"--compare tom_6cls_plain={RES}/rule-eval-tom_6cls_plain")
!.venv/bin/python scripts/score_heldout.py $runs_arg $cmp_arg --folds configs/train/real_check/folds.yaml --out $RES/heldout.md
# 확인: enst_minus_one (64곡)·mdb_drums (23곡) 표, 드러머별 표, 합성 데이터 표, 맨 끝 '## 판정' 아래 네 줄
#   (H1(실제 녹음 데이터)·서비스 개선 가능·H1 페달·H2(학습 설정)). 표의 tom_6cls 줄은 ENST 0.119, MDB 0.744여야 10/31과 같음.
#   '곡 목록이 달라 빠진 곡'이 나오면 그 줄을 적어 둠. 이 출력 전체를 세션에 붙여 줘도 됨

In [ ]:
# 13. 결과 묶기. results 폴더의 real_check_results.zip 을 내려받아 로컬 AI/data/results/ 에 둠 (푸는 것은 세션에서)
#     zip 안 경로는 AI/ 기준 상대 경로 그대로라, 학습마다 같은 이름인 best.pth가 섞이지 않음
import zipfile


def pack(zip_path, paths):
    # 폴더는 안의 파일을 전부, 파일은 그대로. 없는 경로 목록을 돌려줌
    missing = []
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            if os.path.isdir(p):
                for root, _, files in os.walk(p):
                    for f in files:
                        z.write(os.path.join(root, f))
            elif os.path.isfile(p):
                z.write(p)
            else:
                missing.append(p)
    return missing


!rm -rf $RES/train-queue-status && cp -r $RUN_DIR/status $RES/train-queue-status
items = [*(f"{OUT[r]}/{f}" for r in RUNS for f in ("best.pth", "log.csv", "config.yaml")), RES,
         *[os.path.basename(p) for p in (*FEAT_LOG.values(), *RULE_LOG.values(), f"{AI}/adtof_eval_rc.log")]]
missing = pack(f"{RESULTS}/real_check_results.zip", items)
!ls -la $RESULTS/real_check_results.zip
print("zip에 빠진 것:", missing or "없음")
# 확인: 'zip에 빠진 것: 없음'과 수 MB짜리 zip 한 줄. 빠진 것이 있으면 그 셀을 안 친 것이니 치고 다시 13

## H3. 드러머 1 연습 녹음 학습

12번 판정에서 탐이 '아님'으로 나와서 돌린다. `tom_6cls_real_no_d1`에 드러머 1의 반주 없는 연습 연주(hits·phrase·solo)를 더해 학습한다. 드러머 1의 반주 곡 21곡은 여전히 학습에 안 넣고 평가에만 쓴다.

드러머 1 반주 곡에서 정답에 없는 탐(대부분 킥 자리의 플로어 탐)이 크게 줄면, 처음 보는 드럼 세트 소리에 약했던 것이라 여러 세트의 녹음을 모으면 풀린다. 그대로면 데이터를 늘려도 안 되는 것으로 본다.

**순서.** 1 설정 → H3-1 설정 → 1-1 → 4 짧은 시험 → 5 학습 시작 → 6 진행 확인(약 1시간) → 7 특징 저장 → 8 진행 확인 → 9 rule-eval → 9-1 확인 → H3-2 모델 출력 점수 → H3-3 드러머 1 탐 자리 → H3-4 결과 묶기. 커널이 다시 시작됐으면 1번 → H3-1번을 치고 이어 간다. 4·5·7·8·9·9-1번은 H3-1번을 친 뒤에는 이 학습 하나만 다룬다.

In [ ]:
# H3-1. 드러머 1 연습 녹음 학습만 다루게 설정을 바꿈. 1번 셀 바로 다음에 침
PR = "tom_6cls_real_d1_practice"
RUNS = {PR: (f"configs/train/real_check/{PR}.yaml", f"{RC}/adtof_drumsep_{PR}.yaml")}
OUT = {PR: f"data/models/adtof_ft/{PR}"}
FEAT = {PR: f"data/rule_features_rc_{PR}"}
FEAT_LOG = {PR: f"{AI}/rule_features_rc_{PR}.log"}
PARALLEL = 1
R1 = "tom_6cls_real_no_d1"   # 비교 기준. 연습 녹음만 빼고 같은 학습
print("학습", list(RUNS), "· 비교", R1)
!ls -la {RUNS[PR][0]} {RUNS[PR][1]} $DS/real_check/enst_practice_d1.yaml data/models/adtof_ft/$R1/best.pth
!ls -d data/rule_features_rc_$R1 $RES/rule-eval-$R1
# 확인: '학습 ['tom_6cls_real_d1_practice'] · 비교 tom_6cls_real_no_d1', 파일 4줄과 폴더 2줄.
#   'No such file'이면 그 줄을 적어 둠. 코드 zip은 10/31 21시 것 그대로라 다시 올릴 필요 없음.
#   4번 짧은 시험의 학습 곡 수는 tom_6cls_real_no_d1보다 드러머 1 연습 연주 수만큼 많아야 함

In [ ]:
# H3-2. 모델 출력만 본 점수 (GPU, 백그라운드, 5~10분). 학습이 끝난 뒤 한 번만
#      드러머 1 반주 곡과 MDB 1묶음을 연습 녹음 학습과 비교 기준으로 따로따로 채점
import subprocess

assert os.path.isfile(f"{RUN_DIR}/status/{PR}.DONE"), "학습이 아직 안 끝남 (6번 셀)"
EVAL_LOG_H3 = f"{AI}/adtof_eval_rc_h3.log"
assert not os.path.exists(EVAL_LOG_H3), "이미 시작했음. 아래 확인 명령만 다시 침"
cli = ".venv/bin/python -m ssaplash_ai.train.cli adtof-eval"
held1 = f"{DS}/real_check/enst_minus_one_d1.yaml {DS}/real_check/mdb_fold1.yaml"
steps = [f"{cli} --checkpoint {OUT[PR]}/best.pth --dataset {held1} --out {RES}/adtof-eval-heldout-d1-practice",
         f"{cli} --checkpoint {OUT[PR]}/best.pth --dataset {DS}/gmd_test.yaml {DS}/star_drums_test.yaml {DS}/egmd_test_acoustic.yaml --out {RES}/adtof-eval-synth-practice"]
proc = subprocess.Popen(["nohup", "nice", "-n", "19", "bash", "-c", " && ".join(steps)], cwd=AI,
                        stdout=open(EVAL_LOG_H3, "w"), stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
print("시작. pid", proc.pid, "로그", EVAL_LOG_H3)
# 확인: '시작. pid 숫자'. 끝났는지는 아래 셀로

In [ ]:
# H3-2-1. 모델 출력 점수 진행 확인 (여러 번)
!tail -2 $AI/adtof_eval_rc_h3.log
!ls $RES | grep practice
# 확인: ls에 adtof-eval-heldout-d1-practice, adtof-eval-synth-practice 두 폴더가 있고 로그 끝에 'summary.md'가 있으면 끝.
#   'Traceback'이 있으면 로그 끝 20줄을 적어 둠

In [ ]:
# H3-3. ENST drummer_1 정답에 없는 탐 자리 (CPU, nice 19, 5~15분). 7·8번 특징 저장이 끝난 뒤
feats = " ".join(["--features base=data/rule_features", "--features tom_6cls=data/rule_features_ft_tom_6cls",
                  f"--features {R1}=data/rule_features_rc_{R1}", f"--features {PR}={FEAT[PR]}"])
!nice -n 19 .venv/bin/python scripts/check_false_toms.py $feats --dataset $DS/enst_minus_one.yaml --song-prefix drummer_1 --candidates docs/chart-rules/tom-newkits-2026-10-29/candidates.yaml --candidate pk_gap30 --two-tom-upper MID_TOM --out $RES/false-toms-enst-drummer_1-practice.md
# 확인: 곡마다 '[false-toms] drummer_1/…: base 숫자, tom_6cls 숫자, tom_6cls_real_no_d1 숫자, tom_6cls_real_d1_practice 숫자' 21줄, 끝에 '결과 →'.
#   tom_6cls_real_no_d1 숫자를 다 더하면 2,559여야 10/31 결과와 같음. 마지막 숫자가 앞 숫자보다 크게 작으면 연습 녹음이 효과 있는 것

In [ ]:
# H3-4. 결과 묶기. results 폴더의 real_check_h3_results.zip 을 내려받아 로컬 AI/data/results/ 에 둠 (푸는 것은 세션에서)
#      zip 안 경로는 AI/ 기준 상대 경로 그대로
import zipfile


def pack(zip_path, paths):
    # 폴더는 안의 파일을 전부, 파일은 그대로. 없는 경로 목록을 돌려줌
    missing = []
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            if os.path.isdir(p):
                for root, _, files in os.walk(p):
                    for f in files:
                        z.write(os.path.join(root, f))
            elif os.path.isfile(p):
                z.write(p)
            else:
                missing.append(p)
    return missing


!rm -rf $RES/train-queue-status-h3 && cp -r $RUN_DIR/status $RES/train-queue-status-h3
items = [*(f"{OUT[PR]}/{f}" for f in ("best.pth", "log.csv", "config.yaml")),
         f"{RES}/rule-eval-{PR}", f"{RES}/adtof-eval-heldout-d1-practice", f"{RES}/adtof-eval-synth-practice",
         f"{RES}/false-toms-enst-drummer_1-practice.md", f"{RES}/train-queue-status-h3",
         os.path.basename(FEAT_LOG[PR]), os.path.basename(RULE_LOG[PR]), os.path.basename(EVAL_LOG_H3)]
missing = pack(f"{RESULTS}/real_check_h3_results.zip", items)
!ls -la $RESULTS/real_check_h3_results.zip
print("zip에 빠진 것:", missing or "없음")
# 확인: 'zip에 빠진 것: 없음'과 2~3MB짜리 zip 한 줄. 빠진 것이 있으면 그 셀을 안 친 것이니 치고 다시 H3-4
#   EVAL_LOG_H3가 없다는 NameError면 H3-2번 위쪽 EVAL_LOG_H3 = … 한 줄만 먼저 침

## FULL. ENST·MDB 전체로 서비스 후보 학습

`tom_6cls_real_no_d*`와 설정이 같고, 뺀 드러머·묶음 없이 ENST 드러머 1·2·3 전체 연주와 MDB 23곡 전부를 더해 학습한다(`configs/train/real_check/tom_6cls_real_full.yaml`). 학습에 안 쓴 실제 녹음이 남지 않으므로 실제 녹음 점수는 no_d* 세 학습의 합친 결과로 대신하고, 여기서는 합성 데이터 점수가 no_d*와 비슷한지만 본다.

**준비물.** 새 설정 파일이 들어간 코드 zip을 다시 올린다. 로컬 `AI/`에서 `uv run python scripts/pack_server_upload.py --code-only` → `data/upload/server/S15P31A307_code.zip`을 JupyterHub `ssaplash/upload/`에, 이 노트북을 `ssaplash/notebooks/`에 덮어 올린다.

**순서.** 1 설정 → FULL-1 설정 → 1-1 → 2 코드 덮어 풀기 → FULL-1 다시(파일 확인) → 4 짧은 시험 → 5 학습 시작 → 6 진행 확인(약 1~1.5시간) → FULL-2 모델 출력 점수 → FULL-2-1 확인 → FULL-3 결과 묶기. 커널이 다시 시작됐으면 1번 → FULL-1번을 치고 이어 간다. 학습은 새 실행 폴더 `~/ssaplash/runs/164-real-full`의 코드 사본으로 돈다(10/31 폴더의 사본에는 새 설정이 없음).

In [ ]:
# FULL-1. 전체 학습 하나만 다루게 설정을 바꿈. 1번 셀 바로 다음에 침 (2번 셀로 코드를 푼 뒤 한 번 더 쳐서 파일 확인)
FR = "tom_6cls_real_full"
RUNS = {FR: (f"configs/train/real_check/{FR}.yaml", f"{RC}/adtof_drumsep_{FR}.yaml")}
OUT = {FR: f"data/models/adtof_ft/{FR}"}
RUN_DIR = f"{WORK}/runs/164-real-full"   # 새 설정이 든 코드 사본을 쓰려고 새 실행 폴더
PARALLEL = 1
print("학습", list(RUNS), "· 실행 폴더", RUN_DIR)
!ls -la {RUNS[FR][0]} {RUNS[FR][1]} $DS/real_check/enst_train_all.yaml $DS/mdb_drums.yaml
!.venv/bin/ssaplash-bench check-data $DS/real_check/enst_train_all.yaml $DS/mdb_drums.yaml 2>&1 | tail -4
# 확인: '학습 ['tom_6cls_real_full'] · 실행 폴더 …/runs/164-real-full', 파일 4줄.
#   check-data에서 mdb_drums 23곡 누락 0, enst_train_all은 곡 수만 나옴(318 근처, 10/31 3번 셀 합계와 같아야 함).
#   'No such file'이면 코드 zip이 예전 것이니 2번 셀로 새 zip을 풀고 이 셀을 다시 침.
#   4번 짧은 시험의 학습 곡 수는 tom_6cls_real_no_d1(27565개)보다 드러머 1 연주와 MDB 8곡만큼 많아야 함

In [ ]:
# FULL-2. 모델 출력만 본 점수 (GPU, 백그라운드, 10~20분). 학습이 끝난 뒤 한 번만
#        합성 데이터 셋을 전체 학습으로 채점. no_d* 세 학습의 값은 data/results/real_check/adtof-eval-synth 에 있음
import subprocess

assert os.path.isfile(f"{RUN_DIR}/status/{FR}.DONE"), "학습이 아직 안 끝남 (6번 셀)"
EVAL_LOG_FULL = f"{AI}/adtof_eval_rc_full.log"
assert not os.path.exists(EVAL_LOG_FULL), "이미 시작했음. FULL-2-1 셀로 확인"
cli = ".venv/bin/python -m ssaplash_ai.train.cli adtof-eval"
cmd = f"{cli} --checkpoint {OUT[FR]}/best.pth --dataset {DS}/gmd_test.yaml {DS}/star_drums_test.yaml {DS}/egmd_test_acoustic.yaml --out {RES}/adtof-eval-synth-full"
proc = subprocess.Popen(["nohup", "nice", "-n", "19", "bash", "-c", cmd], cwd=AI,
                        stdout=open(EVAL_LOG_FULL, "w"), stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
print("시작. pid", proc.pid, "로그", EVAL_LOG_FULL)
# 확인: '시작. pid 숫자'. 다시 치면 assert에 걸림(정상)

In [ ]:
# FULL-2-1. 모델 출력 점수 진행 확인 (여러 번)
!tail -2 $AI/adtof_eval_rc_full.log
!ls $RES | grep synth
!cat $RUN_DIR/status/{FR}.log | grep -E "에폭 [0-9]+ |문턱값|끝\\." | tail -6
# 확인: ls에 adtof-eval-synth-full 폴더가 있고 로그 끝에 'summary.md'가 있으면 끝.
#   학습 로그는 마지막 에폭 줄 몇 개와 '문턱값 맞춤 …', '끝. data/models/adtof_ft/tom_6cls_real_full/best.pth'.
#   no_d1 비교값: 에폭 10에서 멈춤, 문턱값 맞춤 뒤 검증 f1 0.888·탐 0.739. 비슷하면 정상.
#   'Traceback'이 있으면 로그 끝 20줄을 적어 둠

In [ ]:
# FULL-3. 결과 묶기. results 폴더의 real_full_results.zip 을 내려받아 로컬 AI/data/results/ 에 둠 (푸는 것은 세션에서)
#        zip 안 경로는 AI/ 기준 상대 경로 그대로
import zipfile


def pack(zip_path, paths):
    # 폴더는 안의 파일을 전부, 파일은 그대로. 없는 경로 목록을 돌려줌
    missing = []
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            if os.path.isdir(p):
                for root, _, files in os.walk(p):
                    for f in files:
                        z.write(os.path.join(root, f))
            elif os.path.isfile(p):
                z.write(p)
            else:
                missing.append(p)
    return missing


!rm -rf $RES/train-queue-status-full && cp -r $RUN_DIR/status $RES/train-queue-status-full
items = [*(f"{OUT[FR]}/{f}" for f in ("best.pth", "last.pth", "log.csv", "config.yaml")),
         f"{RES}/adtof-eval-synth-full", f"{RES}/train-queue-status-full", os.path.basename(EVAL_LOG_FULL)]
missing = pack(f"{RESULTS}/real_full_results.zip", items)
!ls -la $RESULTS/real_full_results.zip
print("zip에 빠진 것:", missing or "없음")
# 확인: 'zip에 빠진 것: 없음'과 수 MB짜리 zip 한 줄. 빠진 것이 있으면 그 셀을 안 친 것이니 치고 다시 FULL-3
#   EVAL_LOG_FULL이 없다는 NameError면 FULL-2번 위쪽 EVAL_LOG_FULL = … 한 줄만 먼저 침